[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# *Polars* Avanzado.

* La siguiente celda importa `polars` con el alias convencional `pl`, junto con `pandas`, `numpy`, `os` y los módulos de la biblioteca estándar necesarios para los ejemplos. También define el directorio temporal `data_tmp` para los archivos generados.

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

temp_dir = 'data_tmp'
os.makedirs(temp_dir, exist_ok=True)

print(f"Polars {pl.__version__}")

## *Lazy Evaluation* Profunda.

La **evaluación lazy** es el corazón de la eficiencia de *Polars*. Las operaciones no se ejecutan inmediatamente sino que se construye un **plan de ejecución** que *Polars* puede optimizar antes de materializar ningún dato: aplica *predicate push-down* (mueve los filtros lo antes posible), elimina columnas innecesarias y paraleliza automáticamente.

| Concepto | Descripción |
| :--- | :--- |
| [`df.lazy()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.lazy.html) | Convierte un `DataFrame` en un `LazyFrame`. |
| [`lf.collect()`](https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.LazyFrame.collect.html) | Materializa el `LazyFrame` ejecutando el plan optimizado. |
| [`lf.explain()`](https://docs.pola.rs/api/python/stable/reference/lazyframe/api/polars.LazyFrame.explain.html) | Muestra el plan de ejecución (optimizado o no). |
| [`pl.scan_parquet()`](https://docs.pola.rs/api/python/stable/reference/io/api/polars.scan_parquet.html) | Lee un archivo *Parquet* de forma *lazy*, sin cargarlo en memoria. |
| [`pl.scan_csv()`](https://docs.pola.rs/api/python/stable/reference/io/api/polars.scan_csv.html) | Lee un archivo *CSV* de forma *lazy*. |

https://docs.pola.rs/user-guide/lazy/

**Ejemplos:**

* La siguiente celda crea el `DataFrame` de ejemplo con datos de empleados —id, nombre, departamento, salario y fecha de contrato— que se usará en los ejercicios de *lazy evaluation*.

In [ ]:
df = pl.DataFrame({
    'id': range(1, 101),
    'nombre': [f'persona_{i}' for i in range(100)],
    'departamento': ['Ventas', 'IT', 'HR', 'Finanzas'] * 25,
    'salario': np.random.randint(40000, 120000, 100),
    'fecha_contrato': [datetime(2020, 1, 1) + timedelta(days=x) for x in range(100)]
})

print(f"DataFrame shape: {df.shape}")
print(df.head())

* La siguiente celda define la *query lazy* encadenando `.filter()`, `.select()` y `.sort()`. **Ninguna operación se ejecuta todavía** — *Polars* solo construye el plan de ejecución.

In [ ]:
query = (
    df
    .lazy()
    .filter(pl.col('salario') > 60000)
    .select(['nombre', 'departamento', 'salario'])
    .sort('salario', descending=True)
)

print(f"Tipo: {type(query)}")
print(f"Es lazy: {isinstance(query, pl.LazyFrame)}")

* La siguiente celda imprime el plan de ejecución **sin optimizar** con `explain(optimized=False)`, tal como el usuario lo definió.

In [ ]:
print("Plan sin optimizar:")
print(query.explain(optimized=False))

* La siguiente celda imprime el plan de ejecución **optimizado**: *Polars* aplica *predicate push-down* y elimina columnas innecesarias antes de ejecutar.

In [ ]:
print("Plan optimizado (lo que realmente ejecuta):")
print(query.explain(optimized=True))

* La siguiente celda llama a `.collect()` para materializar el `LazyFrame` en un `DataFrame`, ejecutando el plan optimizado sobre los datos reales.

In [ ]:
result = query.collect()
print(f"Resultado ({len(result)} filas):")
print(result)

### Beneficios de *lazy evaluation*.

* **Optimización automática:** *predicate push-down*, eliminación de columnas no usadas.
* **Memoria eficiente:** solo materializa lo necesario al llamar `.collect()`.
* **Paralelización:** automática en múltiples núcleos.
* **Legibilidad:** el código describe *qué* obtener, no *cómo* calcularlo.

## *Window Functions*.

Las ***window functions*** permiten calcular agregaciones o transformaciones sobre grupos de filas sin reducir el número de filas del resultado, a diferencia de `group_by()`. Se aplican encadenando `.over('columna')` al final de cualquier expresión dentro de `with_columns()`.

| Función | Descripción |
| :--- | :--- |
| [`expr.over("col")`](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.over.html) | Aplica la expresión dentro de cada grupo definido por `col`. |
| [`expr.cum_sum()`](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.cum_sum.html) | Suma acumulada dentro del grupo. |
| [`expr.rank(method=...)`](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.rank.html) | Ranking de valores; métodos: `dense`, `or
| [`expr.shift(n)`](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.shift.html) | Desplaza los valores `n` posiciones (positivo = lag, negativo = lead). |

https://docs.pola.rs/api/python/stable/reference/expressions/window.html

**Ejemplos:**

* La siguiente celda crea el `DataFrame` de ventas por producto y fecha que se usará en los ejemplos de *window functions*.

In [ ]:
ventas = pl.DataFrame({
    'fecha': ['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
              '2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04'] * 3,
    'producto': ['A', 'A', 'A', 'A', 'B', 'B', 'B', 'B',
                 'C', 'C', 'C', 'C', 'A', 'A', 'A', 'A',
                 'B', 'B', 'B', 'B', 'C', 'C', 'C', 'C'],
    'monto': [100, 150, 120, 180, 200, 250, 300, 280,
              75, 95, 110, 130, 110, 160, 130, 190,
              210, 260, 310, 290, 85, 105, 120, 140]
})

print(f"Ventas shape: {ventas.shape}")
print(ventas.head())

* La siguiente celda calcula la suma acumulada de `monto` por producto con `cum_sum().over('producto')`, equivalente a `SUM() OVER (PARTITION BY producto ORDER BY fecha)` en *SQL*.

In [ ]:
resultado = ventas.with_columns(
    pl.col('monto').cum_sum().over('producto').alias('suma_acumulada')
)

print("Con suma acumulada por producto:")
print(resultado.sort(['producto', 'fecha']))

* La siguiente celda calcula el *ranking* de montos por producto. El método `dense` no deja huecos en los valores del rango; `ordinal` asigna posiciones únicas sin empates.

In [ ]:
resultado = ventas.with_columns(
    pl.col('monto').rank(method='dense').over('producto').alias('rank_monto'),
    pl.col('monto').rank(method='ordinal').over('producto').alias('row_number')
)

print("Con ranking por producto:")
print(resultado.sort(['producto', 'monto'], descending=[False, True]))

## *Joins* Optimizados.

*Polars* realiza *joins* de forma muy eficiente mediante *hash joins* paralelizados. La sintaxis general es `df.join(otro, on=..., how=...)`.

| Tipo | `how=` | Descripción |
| :--- | :--- | :--- |
| Inner | `"inner"` | Solo filas con coincidencia en ambos *DataFrames*. |
| Left | `"left"` | Todas las filas del izquierdo; `null` donde no hay coincidencia. |
| Right | `"right"` | Todas las filas del derecho; `null` donde no hay coincidencia. |
| Full | `"full"` | Todas las filas de ambos; `null` donde no hay coincidencia. |
| Semi | `"semi"` | Filas del izquierdo que tienen coincidencia (sin columnas del derecho). |
| Anti | `"anti"` | Filas del izquierdo que **no** tienen coincidencia. |
| Cross | `"cross"` | Producto cartesiano de ambos *DataFrames*. |

https://docs.pola.rs/user-guide/transformations/joins/

**Ejemplos:**

* La siguiente celda crea dos `DataFrame`s relacionados —empleados y salarios— para ilustrar los distintos tipos de *join* disponibles en *Polars*.

In [ ]:
empleados = pl.DataFrame({
    'empleado_id': [1, 2, 3, 4, 5],
    'nombre': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'departamento': ['Ventas', 'IT', 'HR', 'Ventas', 'IT']
})

salarios = pl.DataFrame({
    'empleado_id': [1, 2, 3, 4, 5],
    'salario': [50000, 80000, 45000, 55000, 90000],
    'bonus': [5000, 8000, 0, 5500, 9000]
})

print("Empleados:")
print(empleados)
print("\nSalarios:")
print(salarios)

* La siguiente celda realiza un *inner join* entre `empleados` y `salarios` sobre la columna `empleado_id` con `how='inner'`.

In [ ]:
resultado_inner = empleados.join(
    salarios,
    on='empleado_id',
    how='inner'
)

print("Inner Join:")
print(resultado_inner)

* La siguiente celda realiza un *join* sobre múltiples columnas clave (`ano` y `trimestre`) y calcula una columna derivada (`ganancia`) a partir del resultado.

In [ ]:
tabla_a = pl.DataFrame({
    'ano': [2023, 2023, 2024, 2024],
    'trimestre': [1, 2, 1, 2],
    'ventas': [100, 150, 120, 180]
})

tabla_b = pl.DataFrame({
    'ano': [2023, 2023, 2024, 2024],
    'trimestre': [1, 2, 1, 2],
    'gastos': [60, 80, 70, 100]
})

resultado = tabla_a.join(tabla_b, on=['ano', 'trimestre'], how='inner')
resultado = resultado.with_columns(
    (pl.col('ventas') - pl.col('gastos')).alias('ganancia')
)

print("Join en múltiples columnas con columna calculada:")
print(resultado)

## *Pivoting*.

Las operaciones de *pivoting* transforman datos entre **formato largo** (*long format*, una fila por observación) y **formato ancho** (*wide format*, una columna por categoría).

| Método | Descripción |
| :--- | :--- |
| [`df.pivot()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.pivot.html) | Convierte formato largo a ancho proyectando valores únicos de una columna como nuevas columnas. |
| [`df.unpivot()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.unpivot.html) | Convierte formato ancho a largo (operación inversa de `pivot()`). |

https://docs.pola.rs/user-guide/transformations/pivot/

**Ejemplos:**

* La siguiente celda crea datos de ejemplo en **formato largo** (*long format*): cada fila es una combinación de año, trimestre y departamento — el punto de partida para la transformación con `.pivot()`.

In [ ]:
datos_largo = pl.DataFrame({
    'ano': [2023, 2023, 2023, 2023, 2024, 2024, 2024, 2024],
    'trimestre': ['Q1', 'Q2', 'Q3', 'Q4', 'Q1', 'Q2', 'Q3', 'Q4'],
    'departamento': ['A', 'A', 'B', 'B', 'A', 'A', 'B', 'B'],
    'valor': [100, 150, 200, 180, 120, 160, 210, 190]
})

print("Datos en formato largo:")
print(datos_largo)

* La siguiente celda convierte el formato largo a **ancho** con `.pivot()`, proyectando los valores únicos de `trimestre` como nuevas columnas.

In [ ]:
datos_ancho = datos_largo.pivot(
    index=['ano', 'departamento'],
    columns='trimestre',
    values='valor',
    aggregate_function='first'
)

print("Datos en formato ancho:")
print(datos_ancho)

* La siguiente celda invierte la operación con `.unpivot()`, regresando al formato largo original a partir del formato ancho.

In [ ]:
datos_reconvertidos = datos_ancho.unpivot(
    index=['ano', 'departamento'],
    variable_name='trimestre',
    value_name='valor'
)

print("Reconvertido a formato largo:")
print(datos_reconvertidos.sort(['ano', 'departamento', 'trimestre']))

## I/O de Alto Rendimiento.

*Polars* puede leer archivos de forma **lazy** con las funciones `scan_*`, que solo leen los metadatos hasta que se llama `.collect()`. Esto permite trasladar filtros y selecciones a la capa de lectura (*predicate push-down*), evitando cargar datos innecesarios en memoria.

https://docs.pola.rs/user-guide/io/

**Ejemplos:**

* La siguiente celda escribe un `DataFrame` en *Parquet* con compresión `zstd` y lo lee de forma *lazy* con `pl.scan_parquet()`. El filtro `valor > 0.9` se aplica durante la lectura, sin materializar el archivo completo.

In [ ]:
parquet_url = os.path.join(temp_dir, 'datos_optimizado.parquet')

datos_grandes = pl.DataFrame({
    'id': range(10000),
    'valor': np.random.rand(10000),
    'categoria': np.random.choice(['A', 'B', 'C'], 10000)
})

datos_grandes.write_parquet(parquet_url, compression='zstd')
print("Guardado en Parquet con compresión zstd")

df_lazy = pl.scan_parquet(parquet_url)
print(f"\nLazyFrame escaneado: {type(df_lazy)}")

resultado = df_lazy.filter(pl.col('valor') > 0.9).collect()
print(f"Filas con valor > 0.9: {len(resultado)}")

* La siguiente celda escribe los datos particionados por `categoria` (un archivo *Parquet* por valor único). Al leer con filtro, *Polars* salta las particiones que no coinciden (*partition pruning*).

In [ ]:
parquet_particionado_url = os.path.join(temp_dir, 'datos_particionados')

datos_grandes.write_parquet(
    parquet_particionado_url,
    partition_by='categoria'
)
print("Guardado en Parquet particionado por categoría")

df_particionado = pl.scan_parquet(
    os.path.join(parquet_particionado_url, '**', '*.parquet')
)
resultado = df_particionado.filter(pl.col('categoria') == 'A').collect()
print(f"Filas en categoría A: {len(resultado)}")

* La siguiente celda ejecuta el mismo pipeline —filtro + `group_by` + media— en *Pandas* y *Polars* sobre 5 millones de filas, midiendo el tiempo de cada biblioteca.

In [ ]:
import time

n = 5_000_000
datos = {
    'id': range(n),
    'valor': np.random.rand(n),
    'categoria': np.random.choice(['A', 'B', 'C', 'D'], n)
}

# Benchmark Pandas
start = time.time()
df_pandas = pd.DataFrame(datos)
resultado_pandas = df_pandas[df_pandas['valor'] > 0.5].groupby('categoria')['valor'].mean()
tiempo_pandas = time.time() - start

# Benchmark Polars
start = time.time()
df_polars = pl.DataFrame(datos)
resultado_polars = (
    df_polars
    .filter(pl.col('valor') > 0.5)
    .group_by('categoria')
    .agg(pl.col('valor').mean())
)
tiempo_polars = time.time() - start

print(f"Pandas: {tiempo_pandas:.4f}s")
print(f"Polars: {tiempo_polars:.4f}s")
print(f"Polars es {tiempo_pandas/tiempo_polars:.1f}x más rápido")

## Resumen.

*Polars* avanzado extiende las capacidades del análisis de datos con mecanismos de optimización y transformación que no tienen equivalente directo en *Pandas*:

| Concepto | Descripción |
| :--- | :--- |
| ***Lazy Evaluation*** | Plan de consulta optimizado antes de ejecutar; base del rendimiento de *Polars* |
| **`scan_parquet()` / `scan_csv()`** | Lectura *lazy* de archivos con *predicate push-down* |
| ***Window Functions*** | Agregaciones por grupo sin reducir filas, mediante `.over()` |
| ***Joins*** | *Hash joins* paralelizados con soporte para `inner`, `left`, `semi`, `anti`, etc. |
| ***Pivoting*** | Transformación entre formatos largo y ancho con `pivot()` / `unpivot()` |

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>